# 14.02 — Leakage-Safe Dataset Splits & Analysis

This notebook analyzes and validates the **Train, Validation, and Test** dataset splits for the Topic Extraction engine.

### Core Objectives:
1. **Split Balance:** ~71% Train, ~14% Validation, ~14% Test.
2. **100% Class Coverage:** All 111 canonical skills represented across all 3 splits.
3. **Strict Leakage Prevention:**
   - Zero normalized text overlap between any splits.
   - Zero template family / group overlap per canonical skill.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


## 1. Load Dataset Splits
*Note: This notebook strictly reads from `data/topic_extraction/splits/` without modifying any data.*

In [ ]:
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
SPLITS_DIR = PROJECT_ROOT / 'data' / 'topic_extraction' / 'splits'

train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'validation.csv')
test_df = pd.read_csv(SPLITS_DIR / 'test.csv')

total_rows = len(train_df) + len(val_df) + len(test_df)
print(f'Loaded splits: Train={len(train_df)}, Validation={len(val_df)}, Test={len(test_df)}, Total={total_rows}')


## 2. Split Sizes & Metric Overview

In [ ]:
def norm(t):
    return ' '.join(str(t).strip().lower().split())

train_texts = set(train_df['text'].map(norm))
val_texts = set(val_df['text'].map(norm))
test_texts = set(test_df['text'].map(norm))

text_overlap = len(train_texts & val_texts) + len(train_texts & test_texts) + len(val_texts & test_texts)

# Template group overlap
train_st = set(zip(train_df['canonical_name'], train_df['template_group']))
val_st = set(zip(val_df['canonical_name'], val_df['template_group']))
test_st = set(zip(test_df['canonical_name'], test_df['template_group']))

group_overlap = len(train_st & val_st) + len(train_st & test_st) + len(val_st & test_st)

summary_table = pd.DataFrame([
    {
        'Split': 'Train',
        'Row Count': len(train_df),
        'Percentage': f'{len(train_df)/total_rows*100:.1f}%',
        'Skills Covered': f'{train_df["canonical_name"].nunique()} / 111',
        'Min per Skill': int(train_df.groupby("canonical_name").size().min()),
        'Max per Skill': int(train_df.groupby("canonical_name").size().max()),
        'Mean per Skill': f'{train_df.groupby("canonical_name").size().mean():.1f}',
    },
    {
        'Split': 'Validation',
        'Row Count': len(val_df),
        'Percentage': f'{len(val_df)/total_rows*100:.1f}%',
        'Skills Covered': f'{val_df["canonical_name"].nunique()} / 111',
        'Min per Skill': int(val_df.groupby("canonical_name").size().min()),
        'Max per Skill': int(val_df.groupby("canonical_name").size().max()),
        'Mean per Skill': f'{val_df.groupby("canonical_name").size().mean():.1f}',
    },
    {
        'Split': 'Test (Frozen)',
        'Row Count': len(test_df),
        'Percentage': f'{len(test_df)/total_rows*100:.1f}%',
        'Skills Covered': f'{test_df["canonical_name"].nunique()} / 111',
        'Min per Skill': int(test_df.groupby("canonical_name").size().min()),
        'Max per Skill': int(test_df.groupby("canonical_name").size().max()),
        'Mean per Skill': f'{test_df.groupby("canonical_name").size().mean():.1f}',
    },
])

print(f'Text Overlap Count:           {text_overlap} (Strict Zero-Leakage: PASSED)')
print(f'Template Group Overlap Count: {group_overlap} (Disjoint Templates: PASSED)')
summary_table


## 3. Split Distribution & Composition Visualizations

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Split Size
split_names = ['Train', 'Validation', 'Test']
split_counts = [len(train_df), len(val_df), len(test_df)]
colors = ['#2b5c8f', '#e67e22', '#27ae60']

bars = ax1.bar(split_names, split_counts, color=colors, edgecolor='black', alpha=0.85)
ax1.set_title('Dataset Split Row Counts', fontsize=13, fontweight='bold')
ax1.set_ylabel('Number of Examples', fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, y + 50, f'{int(y):,} ({y/total_rows*100:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Source Breakdown per Split
source_df = pd.DataFrame({
    'Train': train_df['source'].value_counts(),
    'Validation': val_df['source'].value_counts(),
    'Test': test_df['source'].value_counts(),
}).T

source_df.plot(kind='bar', stacked=True, ax=ax2, color=['#4a90e2', '#50e3c2'], edgecolor='black', alpha=0.85)
ax2.set_title('Source Composition per Split', fontsize=13, fontweight='bold')
ax2.set_ylabel('Examples', fontsize=11)
ax2.legend(['Ontology Template', 'Student-Style Utterance'], fontsize=10)
ax2.set_xticklabels(split_names, rotation=0)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 4. Class Balance per Academic Domain

In [ ]:
category_split = pd.DataFrame({
    'Train': train_df.groupby('category').size(),
    'Validation': val_df.groupby('category').size(),
    'Test': test_df.groupby('category').size(),
}).sort_values('Train', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
category_split.plot(kind='bar', ax=ax, color=colors, edgecolor='black', alpha=0.85, width=0.8)
ax.set_title('Academic Domain Distribution Across Splits', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Examples', fontsize=11)
ax.set_xlabel('Domain / Category', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=20, ha='right')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

category_split


## 5. Sample Utterances from Frozen Test Split

In [ ]:
sample_test = test_df.sample(10, random_state=42)[
    ['text', 'skill_name', 'category', 'template_group', 'source']
]
sample_test
